# Model 1: Simple CNN

## 1. Cấu hình và thư viện

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, metrics, callbacks
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
import os
import hashlib
from collections import Counter

DATA_DIR = Path("./data/raw")
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "val"
TEST_DIR = DATA_DIR / "test"

IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 30
NUM_CLASSES = 4

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, 
    image_size=IMG_SIZE, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR, 
    image_size=IMG_SIZE, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    seed=42
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, 
    image_size=IMG_SIZE, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    seed=42
)

class_names = train_ds.class_names
print(f"Các lớp: {class_names}")


In [ ]:
# Tối ưu hóa tốc độ đọc dữ liệu
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## 2. Preprocessing

### 2.1 Loại bỏ ảnh trùng lặp

In [ ]:
SETS = ["train", "val", "test"]
CLASS_NAMES = sorted(p.name for p in (DATA_DIR / "train").iterdir() if p.is_dir())


In [ ]:
CLASS_NAMES

In [ ]:
type(CLASS_NAMES)

In [ ]:
clean_samples = {se: [] for se in SETS}
seen_hash = {}
dup_count = Counter()

for se in ["test", "val", "train"]:
    se_dir = DATA_DIR / se
    
    for class_name in os.listdir(se_dir):
        class_dir = se_dir / class_name
        if not class_dir.is_dir():
            continue
            
        for img_path in class_dir.iterdir():
            if not img_path.is_file():
                continue
            
            # Tính mã MD5 của file ảnh
            with open(img_path, "rb") as f:
                file_hash = hashlib.md5(f.read()).hexdigest()
            
            if file_hash in seen_hash:
                dup_count[se] += 1
            else:
                seen_hash[file_hash] = img_path
                clean_samples[se].append((img_path, class_name))
            
print("Ảnh trùng bị loại:", sum(dup_count.values()))
for (drop_in, kept_in), n in dup_count.items():
    print(f"- {n} ảnh ở '{drop_in}' trùng với ảnh ở '{kept_in}'")
    
del seen_hash # Giải phóng bộ nhớ đệm hash sau khi dùng xong

clean_summary = pd.DataFrame({s: Counter(CLASS_NAMES[y] for _, y in clean_samples[s]) for s in SETS}).fillna(0).astype(int)
clean_summary["total"] = clean_summary.sum(axis=1)
clean_summary.loc["TOTAL"] = clean_summary.sum()
print("\nSau khi làm sạch:")
clean_summary

### 2.2 Kiểm tra mất cân bằng lớp

In [ ]:
# # 2. Trực quan hóa phân bố dữ liệu SAU KHI LỌC (EDA)
# # Lấy danh sách tên lớp từ tập train
# class_names = sorted(os.listdir(DATA_DIR / "train"))

# # Tạo DataFrame từ kết quả đếm để vẽ biểu đồ cho đẹp
# data_summary = []
# for split in SETS:
#     # Đếm số lượng ảnh của từng lớp trong biến clean_samples
#     split_counts = pd.Series([cls for _, cls in clean_samples[split]]).value_counts()
#     for class_name in class_names:
#         count = split_counts.get(class_name, 0)
#         data_summary.append({"Split": split, "Class": class_name, "Count": count})
        
# df_summary = pd.DataFrame(data_summary)

# # Vẽ biểu đồ phân bố
# plt.figure(figsize=(12, 6))
# sns.barplot(data=df_summary, x="Class", y="Count", hue="Split", palette="Set2")
# plt.title("Phân bố số lượng ảnh các lớp (Đã lọc trùng trong bộ nhớ)", fontsize=14, fontweight="bold")
# plt.ylabel("Số lượng ảnh", fontsize=12)
# plt.xlabel("Lớp (Class)", fontsize=12)
# plt.legend(title="Tập dữ liệu")
# plt.tight_layout()
# plt.show()

# # In ra bảng thống kê chi tiết (Pivot table)
# pivot_summary = df_summary.pivot(index="Class", columns="Split", values="Count").fillna(0).astype(int)
# pivot_summary["Total"] = pivot_summary.sum(axis=1)
# print("\n📊 Bảng thống kê chi tiết số lượng ảnh sau khi lọc:")
# display(pivot_summary)

=> Không bị mất cân bằng lớp nặng

## 3. Xây dựng model

In [ ]:
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    # 1. Chuyển pixel từ [0, 255] sang [0, 1]
    layers.Rescaling(1./255),
    
    # 2. Data Augmentation (Keras tự động chỉ áp dụng khi train)
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(15 / 360),
    layers.RandomZoom(
        height_factor=(-0.1, 0.1),
        width_factor=(-0.1, 0.1)
    ),
    layers.RandomBrightness(factor=0.2),
    layers.RandomContrast(factor=0.2),
    layers.RandomSaturation(factor=0.2),
    
    # 3. Normalize theo MEAN và STD của từng kênh RGB
    # layers.Normalization(
    #     axis=-1,
    #     mean=MEAN,
    #     variance=np.square(STD)
    # ),
    
    # CNN    
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.summary()